# Advanced EDA: Data Segmentation Analysis

## Executive Summary & Analytical Domain
This notebook implements multi-dimensional **Data Segmentation** across products, customers, and geographic markets. Segmenting entities into structured tiers enables commercial teams to:
- Understand catalog structure across cost brackets.
- Categorize customer relationships using behavioral Recency, Frequency, Monetary (RFM) and tenure metrics.
- Group geographic territories by overall spending volume for regional resource allocation.

## Segmentation Schemas & Business Rules
1. **Product Cost Segmentation**:
   - `'Below 100'`: `cost < 100`
   - `'100-500'`: `cost BETWEEN 100 AND 500` (Bug fixed: removed accidental leading whitespace).
   - `'500-1000'`: `cost BETWEEN 500 AND 1000`
   - `'Above 1000'`: `cost > 1000`
2. **Customer Behavioral Segmentation**:
   - `'VIP'`: Customer tenure (lifespan) $\ge 12$ months AND lifetime spending $> \text{€}5,000$.
   - `'Regular'`: Customer tenure $\ge 12$ months AND lifetime spending $\le \text{€}5,000$.
   - `'New'`: Customer tenure $< 12$ months.
3. **Regional Market Spending Tiers**:
   - `'High Spending'`: Total territory sales $> 1,000,000$.
   - `'Low Spending'`: Total territory sales $\le 1,000,000$.

## Dialect Translation & Corrections
- **Tenure Calculation**: Computes customer relationship lifespan using Databricks Spark SQL `cast(floor(months_between(max(order_date), min(order_date))) as int)`.
- **Literal Spacing Fix**: Fixed `' 100-500'` to clean `'100-500'`.
- **Column Parity**: References column `cost` on `gold.dim_products`.

## Data Contracts & Layer Dependencies
- **Target Layer**: `gold`
- **Tables Referenced**:
  - `gold.dim_products` (`product_key`, `product_name`, `cost`)
  - `gold.fact_sales` (`order_date`, `sales_amount`, `customer_key`)
  - `gold.dim_customers` (`customer_key`, `country`)

In [ ]:
# Databricks notebook source
# COMMAND ----------
# Setup display helper and session verification
try:
    display
except NameError:
    def display(df, n=20):
        """Fallback display function for standard PySpark and Python environments."""
        if hasattr(df, "show"):
            df.show(n, truncate=False)
        else:
            print(df)

# Ensure gold database context
spark.sql("USE gold")
print("Session configured. Active database context: gold")

## 1. Product Cost Tier Segmentation

### Business Context & Objective
Groups all products into standardized manufacturing cost tiers to understand catalog pricing distribution and assess exposure to low-cost vs luxury tiers.

In [ ]:
# COMMAND ----------
# 1. Product Cost Segmentation
df_product_segments = spark.sql("""
WITH product_segment AS (
    SELECT 
        product_key,
        product_name,
        cost,
        CASE
            WHEN cost < 100 THEN 'Below 100'
            WHEN cost BETWEEN 100 AND 500 THEN '100-500'
            WHEN cost BETWEEN 500 AND 1000 THEN '500-1000'
            ELSE 'Above 1000'
        END AS cost_range
    FROM gold.dim_products
)
SELECT
    cost_range,
    count(product_key) AS total_products
FROM product_segment
GROUP BY cost_range
ORDER BY total_products DESC
""")
display(df_product_segments)

### Strategic Insights (Product Cost Tiers)
- Assesses whether the catalog is predominantly composed of budget accessories (< €100) or high-ticket durable items (> €1000).
- Identifies pricing gaps where intermediate product offerings could capture incremental margin.

## 2. Customer Behavioral Segmentation (VIP, Regular, New)

### Business Context & Objective
Classifies customers based on their relationship duration (months between first and last purchase) and cumulative spend, providing actionable cohorts for targeted marketing campaigns.

In [ ]:
# COMMAND ----------
# 2. Customer Behavioral RFM Segmentation
df_customer_segments = spark.sql("""
WITH customer_spending AS (
    SELECT
        c.customer_key,
        sum(f.sales_amount) AS total_spending,
        min(f.order_date) AS first_order,
        max(f.order_date) AS last_order,
        cast(floor(months_between(max(f.order_date), min(f.order_date))) as int) AS lifespan
    FROM gold.fact_sales f
    LEFT JOIN gold.dim_customers c
        ON f.customer_key = c.customer_key
    GROUP BY c.customer_key
)
SELECT
    customer_segment,
    count(customer_key) AS total_customers
FROM (
    SELECT 
        customer_key,
        total_spending,
        lifespan,
        CASE
            WHEN lifespan >= 12 AND total_spending > 5000 THEN 'VIP'
            WHEN lifespan >= 12 AND total_spending <= 5000 THEN 'Regular'
            ELSE 'New'
        END AS customer_segment
    FROM customer_spending
) cs
GROUP BY customer_segment
ORDER BY total_customers DESC
""")
display(df_customer_segments)

### Strategic Insights (Customer RFM Tiers)
- **VIP Segment**: High-tenure, high-value accounts that generate disproportionate enterprise revenue.
- **Regular Segment**: Loyal, recurring purchasers with potential for upsell and cross-sell expansion.
- **New Segment**: Recently acquired customers requiring structured nurture campaigns to cultivate repeat purchase behavior.

## 3. Geographic Regional Spending Tiers

### Business Context & Objective
Evaluates total spending across geographic countries, categorizing regions into High-Spending and Low-Spending territories.

In [ ]:
# COMMAND ----------
# 3. Regional Spending Tier Analysis
df_regional_tiers = spark.sql("""
SELECT
    c.country,
    CASE
        WHEN sum(f.sales_amount) > 1000000 THEN 'High Spending'
        ELSE 'Low Spending'
    END AS spending_segment,
    sum(f.sales_amount) AS total_spending
FROM gold.fact_sales f
LEFT JOIN gold.dim_customers c
    ON f.customer_key = c.customer_key
GROUP BY c.country
ORDER BY c.country, total_spending DESC
""")
display(df_regional_tiers)

### Business Recommendations & Summary
1. **VIP Loyalty Programs**: Implement exclusive rewards and priority customer service for VIP accounts.
2. **Onboarding Funnels**: Create automated engagement funnels for New customers during their initial 90 days.
3. **Regional Marketing Alignment**: Allocate regional promotional budgets proportionally to high-spending geographic markets.